In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import test_transforms
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir)
transformed_dataset = ImageDataset(annot_path, img_dir, test_transforms)

In [3]:
def collate_fn(batch):
    # unpacks the batch, and then zip creates separate tuples for images and targets
    images, targets = zip(*batch)

    images = torch.stack(images)
    targets = list(targets)

    return images, targets

In [4]:
from torch.utils.data import DataLoader

dl = DataLoader(transformed_dataset, batch_size=4, collate_fn=collate_fn)
dl

In [5]:
X_batch, y_batch = next(iter(dl))

X_batch.shape, len(y_batch)

(torch.Size([4, 3, 224, 224]), 4)

In [6]:
from src.utilities import compute_giou, cxcywh_to_xyxy

bbox_preds = torch.randn(32, 100, 4)

In [7]:
_, truth_labels = next(iter(transformed_dataset))
truth_boxes = truth_labels[:, -4:]
truth_boxes.shape, truth_boxes

(torch.Size([5, 4]),
 tensor([[0.5848, 0.7321, 0.1205, 0.3393],
         [0.4196, 0.8482, 0.1741, 0.2902],
         [0.0714, 0.8259, 0.1250, 0.3482],
         [0.5357, 0.6562, 0.1071, 0.2812],
         [0.5893, 0.5402, 0.0714, 0.0893]]))

In [8]:
giou = compute_giou(truth_boxes, truth_boxes)

In [9]:
giou

tensor([[ 1.0000, -0.2494, -0.6872,  0.0531, -0.6292],
        [-0.3924,  1.0000, -0.4978, -0.4876, -0.9124],
        [-0.7061, -0.4170,  1.0000, -0.7856, -0.9590],
        [ 0.2865, -0.1714, -0.6903,  1.0000, -0.5208],
        [ 0.6577, -0.3058, -0.7199,  0.3854,  1.0000]])